## FieldMeta.jl 开发调试 

In [1]:
using FieldMeta
import FieldMeta: _ispipe, _find_struct, _typname, _fieldname,
                  _strip_leftmost!, _meta_slot, _emit!, _process, REGISTRY

## 1. `_ispipe` —— 是不是 `:|` 表达式？

三个条件：`Expr`、`head === :call`、`args[1] === :|`。

In [2]:
_ispipe(:(a | b)), _ispipe(:(a | b | c)), _ispipe(:(a + b)), _ispipe(:a), _ispipe(1)

(true, true, false, false, false)

## 2. `|` 的左结合性

`a | b | c | d` 解析成 `((a|b)|c)|d`。最左值在最深的 `:|` 节点的 `args[3]`。

In [3]:
ex = :(x::Float64 | (0.01, 0.5) | "m" | "label")
dump(ex; maxdepth=20)

Expr
  head: Symbol call
  args: Array{Any}((3,))
    1: Symbol |
    2: Expr
      head: Symbol call
      args: Array{Any}((3,))
        1: Symbol |
        2: Expr
          head: Symbol call
          args: Array{Any}((3,))
            1: Symbol |
            2: Expr
              head: Symbol ::
              args: Array{Any}((2,))
                1: Symbol x
                2: Symbol Float64
            3: Expr
              head: Symbol tuple
              args: Array{Any}((2,))
                1: Float64 0.01
                2: Float64 0.5
        3: String "m"
    3: String "label"


## 3. `_find_struct` —— 穿过任意层 macrocall 找 struct

用来在 `@with_kw struct ...`、`@some_macro begin struct ... end` 这种包裹下定位真正的 struct 体。

In [4]:
ex = :(@with_kw struct Foo{T}
    x::T = 1.0
    y::T = 2.0
end)
s = _find_struct(ex)
@show s.head s.args[2]

s.head = :struct
s.args[2] = :(Foo{T})


:(Foo{T})

## 4. `_typname` —— 从 struct header 取类型名

支持 `Foo` / `Foo{T}` / `Foo <: A` / `Foo{T} <: A{T}`。

In [5]:
_typname(:Foo), _typname(:(Foo{T})), _typname(:(Foo <: A)), _typname(:(Foo{T} <: A{T}))

(:Foo, :Foo, :Foo, :Foo)

## 5. `_fieldname` —— 穿过 `::`、`=`、`|` 取字段名

递归到底层 Symbol，下游不用关心字段被多少层 wrapper 套着。

In [6]:
_fieldname(:a),
_fieldname(:(a::Int)),
_fieldname(:(a::Int = 3)),
_fieldname(:(a::Int | (0, 1) | "kg" | "label"))

(:a, :a, :a, :a)

## 6. `_strip_leftmost!` —— 整个包最核心的 7 行

把最内层 `:|` 节点替换为它的 lhs，返回 rhs。每次调用相当于一个堆叠宏做的事。

In [7]:
holder = Any[:(x::Float64 | (0.01, 0.5) | "m" | "label")]
v1 = _strip_leftmost!(holder, 1); @show v1 holder[1]
v2 = _strip_leftmost!(holder, 1); @show v2 holder[1]
v3 = _strip_leftmost!(holder, 1); @show v3 holder[1]

v1 = :((0.01, 0.5))
holder[1] = :((x::Float64 | "m") | "label")
v2 = "m"
holder[1] = :(x::Float64 | "label")
v3 = "label"
holder[1] = :(x::Float64)


:(x::Float64)

Form B（`field = default | val`）时，slot 在 `:(=)` 节点的 `args[2]`：

In [8]:
line = :(x::FT = 0.35 | (0.01, 0.5) | "-")
_strip_leftmost!(line.args, 2), _strip_leftmost!(line.args, 2), line

(:((0.01, 0.5)), "-", :(x::FT = 0.35))

## 7. `_meta_slot` —— 把 Form A / Form B 统一抽出 (args, idx, fieldname)

下游 `_strip_leftmost!` 和 `_emit!` 不需要再关心哪种形态。

In [9]:
block = Expr(:block,
    :(a::Int | "x"),
    :(b::Float64),                # 无元数据，跳过
    :(c | "z"),                   # 无类型也可以
    :(d::Float64 = 1.0 | "w"),    # Form B
)
for i in eachindex(block.args)
    println("line $i: ", _meta_slot(block.args, i))
end

line 1: (Any[:(a::Int | "x"), :(b::Float64), :(c | "z"), :(d::Float64 = 1.0 | "w")], 1, :a)
line 2: nothing
line 3: (Any[:(a::Int | "x"), :(b::Float64), :(c | "z"), :(d::Float64 = 1.0 | "w")], 3, :c)
line 4: (Any[:(d::Float64), :(1.0 | "w")], 2, :d)


## 8. 完整宏展开

用 `@macroexpand` 直接观察堆叠宏生成的代码。

In [10]:
@metadata description "" String
@metadata bounds nothing Any

@macroexpand @bounds @description struct Demo
    a::Int | (0, 10) | "label a"
    b::Float64 | _ | "label b"
end

quote
    #= d:\GitHub\jl-pkgs\FieldMeta.jl\docs\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X25sZmlsZQ==.jl:4 =#
    begin
        #= d:\GitHub\jl-pkgs\FieldMeta.jl\docs\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X25sZmlsZQ==.jl:4 =#
        struct Demo
            #= d:\GitHub\jl-pkgs\FieldMeta.jl\docs\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X25sZmlsZQ==.jl:5 =#
            a::Int
            #= d:\GitHub\jl-pkgs\FieldMeta.jl\docs\jl_notebook_cell_df34fa98e69747e1a8f8a730347b8e2f_X25sZmlsZQ==.jl:6 =#
            b::Float64
        end
        begin
            #= d:\GitHub\jl-pkgs\FieldMeta.jl\src\FieldMeta.jl:76 =#
            function (FieldMeta)._meta(::Type{<:Demo}, ::Val{:a}, ::Val{:description})
                $(Expr(:meta, :inline))
                #= d:\GitHub\jl-pkgs\FieldMeta.jl\src\FieldMeta.jl:76 =#
                #= d:\GitHub\jl-pkgs\FieldMeta.jl\src\FieldMeta.jl:77 =#
                v = "label a"
                #= d:\GitHub\jl-pkgs\FieldMeta.j

## 9. 注册表与运行期查找

Accessor 查询路径：`bounds(x, :a)` → `_meta(T, Val{:a}(), Val{:bounds}())` → 命中特化 / 落 fallback 取 `REGISTRY[:bounds][1]`。

In [12]:
REGISTRY

Dict{Symbol, Tuple{Any, Type}} with 2 entries:
  :description => ("", String)
  :bounds      => (nothing, Any)

In [13]:
@bounds @description struct Demo2
    a::Int | (0, 10) | "label a"
    b::Float64 | _ | "label b"
end
d = Demo2(3, 4.0)
@show bounds(d, :a) bounds(d, :b) description(d) description(Demo2)

bounds(d, :a) = (0, 10)
bounds(d, :b) = nothing
description(d) = ("label a", "label b")
description(Demo2) = ("label a", "label b")


("label a", "label b")

## 10. 类型推断检查

In [14]:
using Test
@inferred bounds(Demo2, Val{:a})
@inferred description(d)
@code_warntype bounds(d, :a)

MethodInstance for bounds(::Demo2, ::Symbol)
  from bounds(x, f::Symbol) @ Main d:\GitHub\jl-pkgs\FieldMeta.jl\src\FieldMeta.jl:103
Arguments
  #self#::Core.Const(Main.bounds)
  x::Demo2
  f::Symbol
Body::Any
1 ─ %1  = Base.getproperty(FieldMeta, :_meta)::Core.Const(FieldMeta._meta)
│   %2  = Main.typeof::Core.Const(typeof)
│   %3  = (%2)(x)::Core.Const(Demo2)
│   %4  = Main.Val::Core.Const(Val)
│   %5  = Core.apply_type(%4, f)::Type{Val{_A}} where _A
│   %6  = (%5)()::Val
│   %7  = Main.Val::Core.Const(Val)
│   %8  = Core.apply_type(%7, :bounds)::Core.Const(Val{:bounds})
│   %9  = (%8)()::Core.Const(Val{:bounds}())
│   %10 = (%1)(%3, %6, %9)::Any
└──       return %10

